In [5]:
from pathlib import Path
import json
import pandas as pd

BANDIT_RESULTS_DIR = Path("semgrep-results")

records = []

for issue_file in BANDIT_RESULTS_DIR.glob("*/*/semgrep_introduced_issues.json"):

    repo_dir = issue_file.parent.parent.name
    pr_dir = issue_file.parent.name

    owner_repo = repo_dir.split("__", 1)
    if len(owner_repo) == 2:
        owner, repo = owner_repo
    else:
        owner, repo = None, repo_dir

    pr_number = pr_dir.replace("pr_", "")

    with open(issue_file, "r", encoding="utf-8") as f:
        issues = json.load(f)

    records.append({
        "owner": owner,
        "repo": repo,
        "pr_number": pr_number,
        "issue_file": str(issue_file),
        "introduced_issue_count": len(issues),
        "issues": issues,
    })

pr_df = pd.DataFrame(records)
pr_df = pr_df.sort_values(["introduced_issue_count", "owner", "repo"], ascending=[False, True, True]).reset_index(drop=True)

pr_df.head()

,owner,repo,pr_number,issue_file,introduced_issue_count,issues
0,MontrealAI,AGI-Alpha-Agent-v0,1899,semgrep-results/MontrealAI__AGI-Alpha-Agent-v0...,23,[{'check_id': 'python.lang.security.audit.dyna...
1,wandb,weave,4549,semgrep-results/wandb__weave/pr_4549/semgrep_i...,22,[{'check_id': 'python.django.security.audit.cu...
2,wandb,weave,4607,semgrep-results/wandb__weave/pr_4607/semgrep_i...,20,[{'check_id': 'python.django.security.audit.cu...
3,Azure,azure-sdk-for-python,41852,semgrep-results/Azure__azure-sdk-for-python/pr...,17,[{'check_id': 'python.lang.security.use-defuse...
4,MontrealAI,AGI-Alpha-Agent-v0,2462,semgrep-results/MontrealAI__AGI-Alpha-Agent-v0...,15,[{'check_id': 'python.lang.security.insecure-h...


In [6]:
issue_rows = []

for _, row in pr_df.iterrows():

    for issue in row["issues"]:

        extra = issue.get("extra", {})
        metadata = extra.get("metadata", {})

        issue_rows.append({
            "owner": row["owner"],
            "repo": row["repo"],
            "pr_number": row["pr_number"],

            "rule_id": issue.get("check_id"),
            "message": extra.get("message"),

            "severity": extra.get("severity"),
            "confidence": metadata.get("confidence"),

            "cwe": metadata.get("cwe"),
            "owasp": metadata.get("owasp"),

            "file": issue.get("path"),
            "line": issue.get("start", {}).get("line"),
        })

semgrep_issue_df = pd.DataFrame(issue_rows)

semgrep_issue_df.head()

,owner,repo,pr_number,rule_id,message,severity,confidence,cwe,owasp,file,line
0,MontrealAI,AGI-Alpha-Agent-v0,1899,python.lang.security.audit.dynamic-urllib-use-...,Detected a dynamic value being used with urlli...,WARNING,LOW,[CWE-939: Improper Authorization in Handler fo...,A01:2017 - Injection,alpha_factory_v1/af_requests.py,87
1,MontrealAI,AGI-Alpha-Agent-v0,1899,python.lang.security.audit.dynamic-urllib-use-...,Detected a dynamic value being used with urlli...,WARNING,LOW,[CWE-939: Improper Authorization in Handler fo...,A01:2017 - Injection,alpha_factory_v1/demos/macro_sentinel/data_fee...,71
2,MontrealAI,AGI-Alpha-Agent-v0,1899,python.lang.security.audit.exec-detected.exec-...,Detected the use of exec(). exec() can be dang...,WARNING,LOW,[CWE-95: Improper Neutralization of Directives...,"[A03:2021 - Injection, A05:2025 - Injection]",alpha_factory_v1/demos/meta_agentic_agi/agents...,296
3,MontrealAI,AGI-Alpha-Agent-v0,1899,python.lang.security.audit.exec-detected.exec-...,Detected the use of exec(). exec() can be dang...,WARNING,LOW,[CWE-95: Improper Neutralization of Directives...,"[A03:2021 - Injection, A05:2025 - Injection]",alpha_factory_v1/demos/meta_agentic_agi/agents...,299
4,MontrealAI,AGI-Alpha-Agent-v0,1899,python.lang.security.insecure-hash-algorithms....,Detected SHA1 hash algorithm which is consider...,WARNING,MEDIUM,[CWE-327: Use of a Broken or Risky Cryptograph...,"[A03:2017 - Sensitive Data Exposure, A02:2021 ...",alpha_factory_v1/demos/meta_agentic_agi/core/p...,77


In [8]:
total_semgrep_prs = len(pr_df)
total_semgrep_issues = len(semgrep_issue_df)

print("Semgrep PRs with issues:", total_semgrep_prs)
print("Semgrep total issues:", total_semgrep_issues)

Semgrep PRs with issues: 3372
Semgrep total issues: 571
